## 회원 CRUD - auth.users, user_details
- 삭제 : deleted_at에 날짜와 시간이 업데이트되면 소프트 삭제
- 조회시에는 deleted_at is not null로 조회될 수 있도록 한다.

In [1]:
import os
import uuid
from datetime import datetime, timezone
from dotenv import load_dotenv
from supabase import create_client

# 환경변수 로드
load_dotenv()

url = os.getenv("SUPABASE_URL")
key = os.getenv("SUPABASE_PUBLISHABLE_KEY")

# Supabase 클라이언트 생성
supabase = create_client(url, key)

In [ ]:
class UserService:

    def __init__(self):
        self.table = "user_details" # 사용할 테이블 초기화

    # 회원 상세 정보 등록 (Create)
    # created_at, modified_at, deleted_at 컬럼은 값 자동 입력하므로 생략합니다.
    # 아래 str 자료형은 Python 기준으로 작성되었습니다. (Supabase는 text 자료형으로 생성)
    # `id: str|uuid.UUID` 타입힌팅: 이 변수 자리에는 str이나 uuid.UUID 객체 둘 다 사용할 수 있습니다.
    def create_user_detail(
            self,
            id: str|uuid.UUID,   # PK + refer.auth.users(id)
            type: str = "BUYER", # SELLER or BUYER
            phone: str = None,   # auth의 phone이 아닌 단순 메타데이터
            zipcode: str = None,
            address: str = None,
            address_sub: str = None
    ):

        data = {
            "id": id,
            "type": type,
            "phone": phone,
            "zipcode": zipcode,
            "address": address,
            "address_sub": address_sub, # 데이터를 받습니다.
        }

        response = (
            supabase.table(self.table)
                    .insert(data)
                    .execute()          # 받은 데이터를 DB에 저장합니다.
        )

        return response.data

    # 삭제되지 않은 회원 전체 조회 (Read: Deleted_at IS NULL)
    # supabase-py의 .is_ 매서드는 파이썬의 `IS` 연산자와의 충돌을 막기 위해 언더바를 사용합니다.
    def get_active_users(self):
        response = (
            supabase.table(self.table)
                    .select("*")
                    .is_("deleted_at", "null") # deleted_at 열의 null 값을 탐색합니다.
                    .execute()
        )

        return response.data

    # 삭제되지 않은 단일 회원 조회
    def get_active_user_by_id(self, id: str):
        response = (
            supabase.table(self.table)
                    .select("*")
                    .eq("id", id)
                    .is_("deleted_at", "null")
                    .execute()
        )

        return response.data

    # 수정 가능한 회원 정보 필드
    updatable_fields = {"type", "phone", "zipcode", "address", "address_sub"}

    # 회원 정보 수정 (Update)
    def update_user_detail(self, id: str, **kwargs):
        invalid_keys = set(kwargs.keys()) - self.updatable_fields

        if invalid_keys:
            raise ValueError(f"수정할 수 없는 필드입니다.:{invalid_keys}")

        kwargs["modified_at"] = datetime.now(timezone.utc).isoformat()
        response = (
            supabase.table(self.table)
                    .update(kwargs)             # 수정할 회원정보를 가변키워드인자로 받습니다.
                    .eq("id", id)               # 수정할 회원정보를 가진 회원이 맞는지 확인합니다.
                    .is_("deleted_at", "null")  # 삭제되지 않은 회원임을 확인합니다. 
                    .execute()
        )

        return response.data

    # 소프트 삭제 (deleted_at 업데이트)
    def soft_delete_user(self, id:str|uuid.UUID):
        now = datetime.now(timezone.utc).isoformat() # 현재 시간을 받습니다. 
        response = (
            supabase.table(self.table)
                    .update({"deleted_at":now})      # 받았던 현재시간을 deleted_at 열에 업데이트합니다. ← 삭제 역할!
                    .eq("id", id)                    # 삭제할 회원정보를 가진 회원이 맞는지 확인합니다. 
                    .is_("deleted_at", "null")       # 삭제되지 않은 회원임을 확인합니다. (이제 삭제할겁니다.)
                    .execute()
        )
        return response.data